In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("premierscsv/season-2425.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'premierscsv/season-2425.csv'

In [ ]:
df.dtypes

In [ ]:
df.head

In [447]:
df.shape

(380, 22)

In [448]:
df["HomeTeam"].value_counts()

HomeTeam
Man United        19
Ipswich           19
Arsenal           19
Everton           19
Newcastle         19
Nott'm Forest     19
West Ham          19
Brentford         19
Chelsea           19
Leicester         19
Brighton          19
Crystal Palace    19
Fulham            19
Man City          19
Southampton       19
Tottenham         19
Aston Villa       19
Bournemouth       19
Wolves            19
Liverpool         19
Name: count, dtype: int64

In [449]:
home_teams = df['HomeTeam'].unique()
away_teams = df['AwayTeam'].unique()

In [450]:
home_teams

array(['Man United', 'Ipswich', 'Arsenal', 'Everton', 'Newcastle',
       "Nott'm Forest", 'West Ham', 'Brentford', 'Chelsea', 'Leicester',
       'Brighton', 'Crystal Palace', 'Fulham', 'Man City', 'Southampton',
       'Tottenham', 'Aston Villa', 'Bournemouth', 'Wolves', 'Liverpool'],
      dtype=object)

In [451]:
away_teams

array(['Fulham', 'Liverpool', 'Wolves', 'Brighton', 'Southampton',
       'Bournemouth', 'Aston Villa', 'Crystal Palace', 'Man City',
       'Tottenham', 'Man United', 'West Ham', 'Leicester', 'Ipswich',
       "Nott'm Forest", 'Everton', 'Arsenal', 'Newcastle', 'Chelsea',
       'Brentford'], dtype=object)

In [452]:
teams = pd.unique(pd.concat([pd.Series(home_teams), pd.Series(away_teams)]))

In [453]:
import pandas as pd
import numpy as np

FILE_MAP = {
    "premierscsv/season-2425.csv": 2025,
    "premierscsv/season-2324.csv": 2024,
    "premierscsv/season-2223.csv": 2023,
    "premierscsv/season-2122.csv": 2022,
    "premierscsv/season-2021.csv": 2021
}

FINAL_COLS = [
    'date', 'day', 'comp', 'venue', 'result', 'gf', 'ga', 'opponent',
    'referee', 'sh', 'sot', 'foul_comm', 'foul_rec', 'yc_rec', 'rc_rec',
    'yc_opp', 'rc_opp', 'season', 'team', 'day_code'
]

def get_result(ftr, venue):
    if venue == 'Home':
        if ftr == 'H': return 'W'
        elif ftr == 'A': return 'L'
        else: return 'D'
    elif venue == 'Away':
        if ftr == 'A': return 'W'
        elif ftr == 'H': return 'L'
        else: return 'D'
    return None

def process_season_data(file_name, season_year):
    print(f"Processing {file_name}...")
    try:
        season_df = pd.read_csv(file_name)

        season_df['date'] = pd.to_datetime(season_df['Date'], format='%d/%m/%y').dt.strftime('%Y-%m-%d')
        season_df['season'] = season_year

        home_df = pd.DataFrame({
            'date': season_df['date'],
            'team': season_df['HomeTeam'],
            'opponent': season_df['AwayTeam'],
            'venue': 'Home',
            'gf': season_df['FTHG'],
            'ga': season_df['FTAG'],
            'result': season_df.apply(lambda row: get_result(row['FTR'], 'Home'), axis=1),
            'sh': season_df['HS'],
            'sot': season_df['HST'],
            'referee': season_df['Referee'],
            'season': season_df['season'],
            'foul_comm': season_df['HF'],
            'foul_rec': season_df['AF'],
            'yc_rec': season_df['HY'],
            'rc_rec': season_df['HR'],
            'yc_opp': season_df['AY'],
            'rc_opp': season_df['AR']
        })

        away_df = pd.DataFrame({
            'date': season_df['date'],
            'team': season_df['AwayTeam'],
            'opponent': season_df['HomeTeam'],
            'venue': 'Away',
            'gf': season_df['FTAG'],
            'ga': season_df['FTHG'],
            'result': season_df.apply(lambda row: get_result(row['FTR'], 'Away'), axis=1),
            'sh': season_df['AS'],
            'sot': season_df['AST'],
            'referee': season_df['Referee'],
            'season': season_df['season'],
            'foul_comm': season_df['AF'],
            'foul_rec': season_df['HF'],
            'yc_rec': season_df['AY'],
            'rc_rec': season_df['AR'],
            'yc_opp': season_df['HY'],
            'rc_opp': season_df['HR']
        })

        combined_df = pd.concat([home_df, away_df], ignore_index=True)

        combined_df['comp'] = 'Premier League'
        combined_df['date_dt'] = pd.to_datetime(combined_df['date'])
        combined_df['day'] = combined_df['date_dt'].dt.day_name().str[:3]
        combined_df['day_code'] = combined_df['date_dt'].dt.dayofweek
        combined_df = combined_df.drop(columns=['date_dt'])

        float_cols = [
            'gf', 'ga', 'sh', 'sot', 'foul_comm', 'foul_rec', 'yc_rec', 'rc_rec', 'yc_opp', 'rc_opp'
        ]
        for col in float_cols:
            combined_df[col] = combined_df[col].astype(float)

        final_df = combined_df[FINAL_COLS].copy()

        return final_df
    except Exception as e:
        print(f"Error processing file {file_name}: {e}")
        return pd.DataFrame(columns=FINAL_COLS)

all_seasons_data = []

for file_name, season_year in FILE_MAP.items():
    processed_df = process_season_data(file_name, season_year)
    all_seasons_data.append(processed_df)

combined_data = pd.concat(all_seasons_data, ignore_index=True)

combined_data.sort_values(by=['date', 'team'], inplace=True, ignore_index=True)

combined_file_name = 'premierscsv/all_seasons_combined_sorted.csv'
combined_data.to_csv(combined_file_name, index=False)

print(f"\nSuccessfully combined all data into: {combined_file_name}")
print(f"Total rows in combined data: {len(combined_data)}")

Processing premierscsv/season-2425.csv...
Processing premierscsv/season-2324.csv...
Processing premierscsv/season-2223.csv...
Processing premierscsv/season-2122.csv...
Processing premierscsv/season-2021.csv...

Successfully combined all data into: premierscsv/all_seasons_combined_sorted.csv
Total rows in combined data: 3800


In [454]:
combined_data

,date,day,comp,venue,result,gf,ga,opponent,referee,sh,sot,foul_comm,foul_rec,yc_rec,rc_rec,yc_opp,rc_opp,season,team,day_code
0,2020-09-12,Sat,Premier League,Away,W,3.0,0.0,Fulham,C Kavanagh,13.0,6.0,12.0,12.0,2.0,0.0,2.0,0.0,2021,Arsenal,5
1,2020-09-12,Sat,Premier League,Home,W,1.0,0.0,Southampton,J Moss,5.0,3.0,14.0,11.0,2.0,0.0,1.0,0.0,2021,Crystal Palace,5
2,2020-09-12,Sat,Premier League,Home,L,0.0,3.0,Arsenal,C Kavanagh,5.0,2.0,12.0,12.0,2.0,0.0,2.0,0.0,2021,Fulham,5
3,2020-09-12,Sat,Premier League,Away,L,3.0,4.0,Liverpool,M Oliver,6.0,3.0,6.0,9.0,0.0,0.0,1.0,0.0,2021,Leeds,5
4,2020-09-12,Sat,Premier League,Home,W,4.0,3.0,Leeds,M Oliver,22.0,6.0,9.0,6.0,1.0,0.0,0.0,0.0,2021,Liverpool,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3795,2025-05-25,Sun,Premier League,Home,L,0.0,1.0,Chelsea,A Taylor,10.0,2.0,10.0,11.0,2.0,0.0,2.0,0.0,2025,Nott'm Forest,6
3796,2025-05-25,Sun,Premier League,Home,L,1.0,2.0,Arsenal,D Bond,7.0,2.0,7.0,8.0,0.0,0.0,0.0,0.0,2025,Southampton,6
3797,2025-05-25,Sun,Premier League,Home,L,1.0,4.0,Brighton,R Jones,4.0,2.0,13.0,8.0,3.0,0.0,1.0,0.0,2025,Tottenham,6
3798,2025-05-25,Sun,Premier League,Away,W,3.0,1.0,Ipswich,T Robinson,10.0,6.0,9.0,10.0,1.0,0.0,1.0,0.0,2025,West Ham,6


In [455]:
combined_data["venue_code"] = combined_data["venue"].astype("category").cat.codes

In [456]:
combined_data["opp_code"] = combined_data["opponent"].astype("category").cat.codes

In [457]:
combined_data["day_code"] = pd.to_datetime(combined_data["date"]).dt.dayofweek

In [458]:
combined_data["target"] = (combined_data["result"] == "W").astype("int")

In [459]:
combined_data

,date,day,comp,venue,result,gf,ga,opponent,referee,sh,...,yc_rec,rc_rec,yc_opp,rc_opp,season,team,day_code,venue_code,opp_code,target
0,2020-09-12,Sat,Premier League,Away,W,3.0,0.0,Fulham,C Kavanagh,13.0,...,2.0,0.0,2.0,0.0,2021,Arsenal,5,0,9,1
1,2020-09-12,Sat,Premier League,Home,W,1.0,0.0,Southampton,J Moss,5.0,...,2.0,0.0,1.0,0.0,2021,Crystal Palace,5,1,21,1
2,2020-09-12,Sat,Premier League,Home,L,0.0,3.0,Arsenal,C Kavanagh,5.0,...,2.0,0.0,2.0,0.0,2021,Fulham,5,1,0,0
3,2020-09-12,Sat,Premier League,Away,L,3.0,4.0,Liverpool,M Oliver,6.0,...,0.0,0.0,1.0,0.0,2021,Leeds,5,0,13,0
4,2020-09-12,Sat,Premier League,Home,W,4.0,3.0,Leeds,M Oliver,22.0,...,1.0,0.0,0.0,0.0,2021,Liverpool,5,1,11,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3795,2025-05-25,Sun,Premier League,Home,L,0.0,1.0,Chelsea,A Taylor,10.0,...,2.0,0.0,2.0,0.0,2025,Nott'm Forest,6,1,6,0
3796,2025-05-25,Sun,Premier League,Home,L,1.0,2.0,Arsenal,D Bond,7.0,...,0.0,0.0,0.0,0.0,2025,Southampton,6,1,0,0
3797,2025-05-25,Sun,Premier League,Home,L,1.0,4.0,Brighton,R Jones,4.0,...,3.0,0.0,1.0,0.0,2025,Tottenham,6,1,4,0
3798,2025-05-25,Sun,Premier League,Away,W,3.0,1.0,Ipswich,T Robinson,10.0,...,1.0,0.0,1.0,0.0,2025,West Ham,6,0,10,1


In [460]:
from sklearn.ensemble import RandomForestClassifier

In [461]:
rf = RandomForestClassifier(n_estimators = 40, min_samples_split = 300, random_state=1)

In [462]:
train = combined_data[combined_data["date"] < '2023-01-01']

In [463]:
test = combined_data[combined_data["date"] > '2023-01-01']

In [464]:
predictors = ["venue_code", "opp_code", "day_code"]

In [465]:
rf.fit(train[predictors], train["target"])

,n_estimators,40
,criterion,'gini'
,max_depth,None
,min_samples_split,300
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [466]:
preds = rf.predict(test[predictors])

In [467]:
from sklearn.metrics import accuracy_score

In [468]:
acc = accuracy_score(test["target"], preds)

In [469]:
acc

0.6267967145790554

In [470]:
combined = pd.DataFrame(dict(actual=test["target"], prediction=preds))

In [471]:
pd.crosstab(index=combined["actual"], columns=combined["prediction"])

prediction,0,1
actual,,
0,1120,78
1,649,101


In [472]:
from sklearn.metrics import precision_score

In [473]:
precision_score(test["target"], preds)

0.5642458100558659

In [474]:
grouped_matches = combined_data.groupby("team")

In [475]:
group = grouped_matches.get_group("Man United")

In [476]:
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

In [477]:
cols = ["gf","ga","sh","sot"]
new_cols = [f"{c}_rolling" for c in cols]

In [478]:
new_cols

['gf_rolling', 'ga_rolling', 'sh_rolling', 'sot_rolling']

In [479]:
combined_data.dtypes

date           object
day            object
comp           object
venue          object
result         object
gf            float64
ga            float64
opponent       object
referee        object
sh            float64
sot           float64
foul_comm     float64
foul_rec      float64
yc_rec        float64
rc_rec        float64
yc_opp        float64
rc_opp        float64
season          int64
team           object
day_code        int32
venue_code       int8
opp_code         int8
target          int64
dtype: object

In [480]:
rolling_averages(group,cols,new_cols)

,date,day,comp,venue,result,gf,ga,opponent,referee,sh,...,season,team,day_code,venue_code,opp_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling
81,2020-10-17,Sat,Premier League,Away,W,4.0,1.0,Newcastle,C Pawson,28.0,...,2021,Man United,5,0,17,1,1.666667,3.666667,8.333333,3.000000
103,2020-10-24,Sat,Premier League,Home,D,0.0,0.0,Chelsea,M Atkinson,14.0,...,2021,Man United,5,1,6,0,2.666667,3.000000,13.333333,6.333333
128,2020-11-01,Sun,Premier League,Home,L,0.0,1.0,Arsenal,M Dean,8.0,...,2021,Man United,6,1,0,0,1.666667,2.333333,15.666667,6.666667
145,2020-11-07,Sat,Premier League,Away,W,3.0,1.0,Everton,P Tierney,13.0,...,2021,Man United,5,0,8,1,1.333333,0.666667,16.666667,6.666667
160,2020-11-21,Sat,Premier League,Home,W,1.0,0.0,West Brom,D Coote,17.0,...,2021,Man United,5,1,24,1,1.000000,0.666667,11.666667,3.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3716,2025-04-27,Sun,Premier League,Away,D,1.0,1.0,Bournemouth,P Bankes,25.0,...,2025,Man United,6,0,2,0,0.333333,1.666667,11.333333,3.000000
3734,2025-05-04,Sun,Premier League,Away,L,3.0,4.0,Brentford,A Taylor,14.0,...,2025,Man United,6,0,3,0,0.666667,2.000000,15.333333,4.333333
3755,2025-05-11,Sun,Premier League,Home,L,0.0,2.0,West Ham,J Gillett,20.0,...,2025,Man United,6,1,25,0,1.333333,2.000000,17.000000,4.333333
3762,2025-05-16,Fri,Premier League,Away,L,0.0,1.0,Chelsea,C Kavanagh,4.0,...,2025,Man United,4,0,6,0,1.333333,2.333333,19.666667,5.333333


In [481]:
matches_rolling = combined_data.groupby("team").apply(lambda x: rolling_averages(x,cols,new_cols))

/var/folders/59/n670c_993y1708914z3l06zr0000gq/T/ipykernel_77294/633759359.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = combined_data.groupby("team").apply(lambda x: rolling_averages(x,cols,new_cols))


In [482]:
matches_rolling

date  day            comp venue result   gf   ga  \
team                                                                   
Arsenal 64    2020-10-04  Sun  Premier League  Home      W  2.0  1.0   
        76    2020-10-17  Sat  Premier League  Away      L  0.0  1.0   
        106   2020-10-25  Sun  Premier League  Home      L  0.0  1.0   
        124   2020-11-01  Sun  Premier League  Away      W  1.0  0.0   
        148   2020-11-08  Sun  Premier League  Home      L  0.0  3.0   
...                  ...  ...             ...   ...    ...  ...  ...   
Wolves  3713  2025-04-26  Sat  Premier League  Home      W  3.0  0.0   
        3721  2025-05-02  Fri  Premier League  Away      L  0.0  1.0   
        3749  2025-05-10  Sat  Premier League  Home      L  0.0  2.0   
        3779  2025-05-20  Tue  Premier League  Away      L  2.0  4.0   
        3799  2025-05-25  Sun  Premier League  Home      D  1.0  1.0   

                      opponent     referee    sh  ...  season     team  \
team                                              ...                    
Arsenal 64    Sheffield United     L Mason   6.0  ...    2021  Arsenal   
        76            Man City  C Kavanagh  11.0  ...    2021  Arsenal   
        106          Leicester    C Pawson  12.0  ...    2021  Arsenal   
        124         Man United      M Dean   7.0  ...    2021  Arsenal   
        148        Aston Villa  M Atkinson  13.0  ...    2021  Arsenal   
...                        ...         ...   ...  ...     ...      ...   
Wolves  3713         Leicester   S Barrott  20.0  ...    2025   Wolves   
        3721          Man City    P Bankes   6.0  ...    2025   Wolves   
        3749          Brighton    M Oliver  10.0  ...    2025   Wolves   
        3779    Crystal Palace      D Webb  12.0  ...    2025   Wolves   
        3799         Brentford    C Pawson  18.0  ...    2025   Wolves   

              day_code  venue_code  opp_code  target  gf_rolling  ga_rolling  \
team                                                                           
Arsenal 64           6           1        20       1    2.000000    1.333333   
        76           5           0        15       0    1.666667    1.666667   
        106          6           1        12       0    1.000000    1.666667   
        124          6           0        16       1    0.666667    1.000000   
        148          6           1         1       0    0.333333    0.666667   
...                ...         ...       ...     ...         ...         ...   
Wolves  3713         5           1        12       1    2.333333    1.000000   
        3721         4           0        15       0    2.666667    0.666667   
        3749         5           1         4       0    1.333333    0.333333   
        3779         1           0         7       0    1.000000    1.000000   
        3799         6           1         3       0    0.666667    2.333333   

             sh_rolling  sot_rolling  
team                                  
Arsenal 64     8.000000     4.000000  
        76     5.666667     3.666667  
        106    7.000000     3.666667  
        124    9.666667     4.000000  
        148   10.000000     3.000000  
...                 ...          ...  
Wolves  3713  13.000000     4.666667  
        3721  12.333333     4.333333  
        3749  10.000000     3.000000  
        3779  12.000000     3.333333  
        3799   9.333333     2.333333  

[3719 rows x 27 columns]

In [483]:
matches_rolling = matches_rolling.droplevel('team')

In [484]:
matches_rolling

,date,day,comp,venue,result,gf,ga,opponent,referee,sh,...,season,team,day_code,venue_code,opp_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling
64,2020-10-04,Sun,Premier League,Home,W,2.0,1.0,Sheffield United,L Mason,6.0,...,2021,Arsenal,6,1,20,1,2.000000,1.333333,8.000000,4.000000
76,2020-10-17,Sat,Premier League,Away,L,0.0,1.0,Man City,C Kavanagh,11.0,...,2021,Arsenal,5,0,15,0,1.666667,1.666667,5.666667,3.666667
106,2020-10-25,Sun,Premier League,Home,L,0.0,1.0,Leicester,C Pawson,12.0,...,2021,Arsenal,6,1,12,0,1.000000,1.666667,7.000000,3.666667
124,2020-11-01,Sun,Premier League,Away,W,1.0,0.0,Man United,M Dean,7.0,...,2021,Arsenal,6,0,16,1,0.666667,1.000000,9.666667,4.000000
148,2020-11-08,Sun,Premier League,Home,L,0.0,3.0,Aston Villa,M Atkinson,13.0,...,2021,Arsenal,6,1,1,0,0.333333,0.666667,10.000000,3.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3713,2025-04-26,Sat,Premier League,Home,W,3.0,0.0,Leicester,S Barrott,20.0,...,2025,Wolves,5,1,12,1,2.333333,1.000000,13.000000,4.666667
3721,2025-05-02,Fri,Premier League,Away,L,0.0,1.0,Man City,P Bankes,6.0,...,2025,Wolves,4,0,15,0,2.666667,0.666667,12.333333,4.333333
3749,2025-05-10,Sat,Premier League,Home,L,0.0,2.0,Brighton,M Oliver,10.0,...,2025,Wolves,5,1,4,0,1.333333,0.333333,10.000000,3.000000
3779,2025-05-20,Tue,Premier League,Away,L,2.0,4.0,Crystal Palace,D Webb,12.0,...,2025,Wolves,1,0,7,0,1.000000,1.000000,12.000000,3.333333


In [485]:
matches_rolling.index = range(matches_rolling.shape[0])

In [486]:
matches_rolling

,date,day,comp,venue,result,gf,ga,opponent,referee,sh,...,season,team,day_code,venue_code,opp_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling
0,2020-10-04,Sun,Premier League,Home,W,2.0,1.0,Sheffield United,L Mason,6.0,...,2021,Arsenal,6,1,20,1,2.000000,1.333333,8.000000,4.000000
1,2020-10-17,Sat,Premier League,Away,L,0.0,1.0,Man City,C Kavanagh,11.0,...,2021,Arsenal,5,0,15,0,1.666667,1.666667,5.666667,3.666667
2,2020-10-25,Sun,Premier League,Home,L,0.0,1.0,Leicester,C Pawson,12.0,...,2021,Arsenal,6,1,12,0,1.000000,1.666667,7.000000,3.666667
3,2020-11-01,Sun,Premier League,Away,W,1.0,0.0,Man United,M Dean,7.0,...,2021,Arsenal,6,0,16,1,0.666667,1.000000,9.666667,4.000000
4,2020-11-08,Sun,Premier League,Home,L,0.0,3.0,Aston Villa,M Atkinson,13.0,...,2021,Arsenal,6,1,1,0,0.333333,0.666667,10.000000,3.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3714,2025-04-26,Sat,Premier League,Home,W,3.0,0.0,Leicester,S Barrott,20.0,...,2025,Wolves,5,1,12,1,2.333333,1.000000,13.000000,4.666667
3715,2025-05-02,Fri,Premier League,Away,L,0.0,1.0,Man City,P Bankes,6.0,...,2025,Wolves,4,0,15,0,2.666667,0.666667,12.333333,4.333333
3716,2025-05-10,Sat,Premier League,Home,L,0.0,2.0,Brighton,M Oliver,10.0,...,2025,Wolves,5,1,4,0,1.333333,0.333333,10.000000,3.000000
3717,2025-05-20,Tue,Premier League,Away,L,2.0,4.0,Crystal Palace,D Webb,12.0,...,2025,Wolves,1,0,7,0,1.000000,1.000000,12.000000,3.333333


In [487]:
def make_predictions(data,predictors):
    train = data[data["date"] < '2023-01-01']
    test = data[data["date"] > '2023-01-01']
    rf.fit(train[predictors], train["target"])
    preds = rf.predict(test[predictors])
    combined = pd.DataFrame(dict(actual = test["target"], predicted = preds), index = test.index)
    precision = precision_score(test["target"], preds)
    return combined, precision


In [488]:
combined, precision = make_predictions(matches_rolling, predictors + new_cols)


In [489]:
precision

0.6551724137931034

In [490]:
combined = combined.merge(matches_rolling[["date","team","opponent", "result"]], left_index=True, right_index=True)

In [491]:
class MissingDict(dict):
    __missing__ = lambda self, key: key

map_values = {"Brighton": "Brighton"}
mapping = MissingDict(**map_values)

In [492]:
combined["new_team"] = combined["team"].map(mapping)


In [493]:
combined

,actual,predicted,date,team,opponent,result,new_team
89,0,0,2023-01-03,Arsenal,Newcastle,D,Arsenal
90,1,1,2023-01-15,Arsenal,Tottenham,W,Arsenal
91,1,0,2023-01-22,Arsenal,Man United,W,Arsenal
92,0,0,2023-02-04,Arsenal,Everton,L,Arsenal
93,0,0,2023-02-11,Arsenal,Brentford,D,Arsenal
...,...,...,...,...,...,...,...
3714,1,0,2025-04-26,Wolves,Leicester,W,Wolves
3715,0,0,2025-05-02,Wolves,Man City,L,Wolves
3716,0,0,2025-05-10,Wolves,Brighton,L,Wolves
3717,0,0,2025-05-20,Wolves,Crystal Palace,L,Wolves


In [494]:
merged = combined.merge(combined, left_on=["date", "new_team"], right_on=["date", "opponent"])


In [495]:
merged[(merged["predicted_x"] == 1) & (merged["predicted_y"] ==0)]["actual_x"].value_counts()

actual_x
1    129
0     66
Name: count, dtype: int64

In [496]:
merged.to_csv("premierscsv/merged.csv",index=False)

In [497]:
import pandas as pd
import numpy as np

POINTS_FOR_WIN = 3
POINTS_FOR_DRAW = 1
POINTS_FOR_LOSS = 0
GAMES_PER_PREMIER_LEAGUE_SEASON = 38 
TOTAL_FIXTURES_PER_SEASON = 380 

DRAW_PROBABILITY_ON_ZERO = 0.25

def simulate_league_cumulative(df):
    print(f"--- Running Cumulative Simulation based on {len(df)} Fixtures ---")

    home_teams = df['team_x'].unique()
    away_teams = df['opponent_x'].unique()
    all_teams = sorted(list(set(list(home_teams) + list(away_teams))))

    league_table = {}
    for team in all_teams:
        league_table[team] = {
            'Points': 0,
            'Played': 0,
            'Wins': 0,
            'Draws': 0,
            'Losses': 0,        }

    for index, row in df.iterrows():
        home_team = row['team_x']
        away_team = row['opponent_x']
        prediction = int(row['predicted_x']) 

        league_table[home_team]['Played'] += 1
        league_table[away_team]['Played'] += 1

        if prediction == 1:
            league_table[home_team]['Points'] += POINTS_FOR_WIN
            league_table[home_team]['Wins'] += 1
            league_table[away_team]['Losses'] += 1

        elif prediction == 0:
           
            if np.random.rand() < DRAW_PROBABILITY_ON_ZERO:
                league_table[home_team]['Draws'] += 1
                league_table[home_team]['Points'] += POINTS_FOR_DRAW
                
                league_table[away_team]['Draws'] += 1
                league_table[away_team]['Points'] += POINTS_FOR_DRAW
            else:
                league_table[home_team]['Losses'] += 1
                
                league_table[away_team]['Points'] += POINTS_FOR_WIN
                league_table[away_team]['Wins'] += 1


    final_table = pd.DataFrame.from_dict(league_table, orient='index')
    final_table = final_table[final_table['Played'] > 0]
    
    return final_table


try:
    df_predictions = pd.read_csv('premierscsv/merged.csv')

    df_predictions.columns = df_predictions.columns.str.strip().str.lower()
    
    required_cols_lower = ['team_x', 'opponent_x', 'predicted_x']
    if not all(col in df_predictions.columns for col in required_cols_lower):
        raise ValueError(f"CSV file must contain columns: {required_cols_lower}")

    cumulative_table = simulate_league_cumulative(df_predictions)

    if cumulative_table.empty:
        print("\nNo teams found in the prediction data.")
        raise SystemExit(0) 
        
    averaged_table = cumulative_table.copy()

    normalization_factor = averaged_table['Played'] / GAMES_PER_PREMIER_LEAGUE_SEASON
    
    print(f"\nNOTE: Normalizing each team's statistics to an average single {GAMES_PER_PREMIER_LEAGUE_SEASON}-match season.")

    cols_to_normalize = ['Wins', 'Draws', 'Losses', 'Points']
    
    for col in cols_to_normalize:
        new_col_name = f'Avg. {col}'
        averaged_table[new_col_name] = averaged_table[col].div(normalization_factor).fillna(0) 

    averaged_table['Avg. Played'] = float(GAMES_PER_PREMIER_LEAGUE_SEASON)
    final_cols = ['Avg. Played', 'Avg. Wins', 'Avg. Draws', 'Avg. Losses', 'Avg. Points']
    final_table = averaged_table[final_cols].copy()

    final_table = final_table.sort_values(
        by=['Avg. Points', 'Avg. Wins'], 
        ascending=[False, False]
    )

    final_table_display = final_table.round(2)
    
    max_played = cumulative_table['Played'].max()
    num_seasons_data = round(max_played / GAMES_PER_PREMIER_LEAGUE_SEASON)
    if num_seasons_data < 1: num_seasons_data = 1

    print("\n" + "="*70)
    print(f"  PREDICTED PREMIER LEAGUE TABLE (Averaged over Data Span of {num_seasons_data} Seasons)")
    print("="*70)
    print(final_table_display.to_string())
    print("="*70)

    if not final_table.empty:
        print(f"\n--- Average Predicted Standings ---\n")
        avg_champ = final_table.index[0]
        avg_points = final_table_display['Avg. Points'].iloc[0] 
        print(f"Predicted Champion (Average Season): {avg_champ} with {avg_points} points.")
    
        if len(final_table) >= 20:
             relegation_teams = final_table.index[-3:].tolist()
             print(f"Predicted Relegation (Average Season): {', '.join(relegation_teams)}.")

        champions_league = final_table.index[:4].tolist()
        print(f"Predicted UEFA Champions League Group Stage (Average Season): {', '.join(champions_league)}.")

        europa_league= final_table.index[5]
        europa_league2 = final_table.index[6]
        print(f"Predicted Europa League Group Stage (Average Season): {europa_league}, {europa_league2}")
        

except FileNotFoundError:
    print(f"Error: The file 'merged.csv' was not found.")
    print("Please make sure 'merged.csv' is in the same directory as this script.")
except ValueError as e:
    print(f"Data Error: {e}")
except SystemExit:
    pass
except Exception as e:
    print(f"An unexpected error occurred: {e}")


--- Running Cumulative Simulation based on 1936 Fixtures ---

NOTE: Normalizing each team's statistics to an average single 38-match season.

  PREDICTED PREMIER LEAGUE TABLE (Averaged over Data Span of 5 Seasons)
                  Avg. Played  Avg. Wins  Avg. Draws  Avg. Losses  Avg. Points
Liverpool                38.0      21.74        7.25         9.01        72.47
Man City                 38.0      21.15        8.03         8.81        71.49
Brighton                 38.0      16.65        8.81        12.54        58.76
Newcastle                38.0      16.26        8.03        13.71        56.80
Arsenal                  38.0      15.70        9.31        12.99        56.42
Aston Villa              38.0      15.47        9.99        12.54        56.41
Man United               38.0      15.32       10.28        12.41        56.22
Crystal Palace           38.0      15.12        9.50        13.38        54.87
Chelsea                  38.0      14.49        9.99        13.52        53